# Review Sentiment Lab: exploratory data analysis
Run from the repository environment. This notebook does not train or save models. Explore training data only to avoid adapting decisions to validation/test evidence.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from src.config import load_config
from src.data import load_data, split_data

In [ ]:
config = load_config(ROOT / "configs/default.yaml")
cleaned, audit = load_data(config["paths"]["data"])
train, _, _ = split_data(cleaned, config["seed"])
pd.Series({k: v for k, v in audit.items() if k.endswith("removed")})

In [ ]:
pd.crosstab(train.source, train.label).plot.bar(title="Training class balance by source")
plt.tight_layout()

In [ ]:
train.assign(length=train.review_text.str.len()).boxplot(column="length", by="label", showfliers=False)
plt.suptitle("Training review length by rating-derived sentiment")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from src.preprocess import clean_text
vectorizer = CountVectorizer(preprocessor=clean_text, max_features=30)
counts = vectorizer.fit_transform(train.review_text)
pd.Series(counts.sum(axis=0).A1, index=vectorizer.get_feature_names_out()).sort_values().plot.barh(figsize=(8, 9))

## Interpretation
Record observations after execution. Check class imbalance, source imbalance and review length. Frequent terms describe this corpus, not causal drivers. Never copy notebook numbers into the README; use generated reports. Training and evaluation belong in the offline CLI.